# CareTrace RAGAS Clinical Evaluation

This notebook demonstrates how to use the RAGAS framework to evaluate CareTrace.

## RAGAS Metrics (adapted for clinical triage)

- **Extraction Accuracy**: Were clinical fields (temp, age, alertness) extracted correctly?
- **Faithfulness**: Is disposition grounded in fired rules and extracted data?
- **Answer Relevance**: Is disposition clinically appropriate and actionable?
- **Context Relevance**: Are KG annotations relevant to the case?
- **Context Recall**: Were all relevant KG concepts retrieved?
- **Context Precision**: Are retrieved KG facts high-quality?
- **RAGAS Score**: Composite (0-1)
- **Clinical Score**: Clinical-focused variant (0-1)

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Import RAGAS evaluation framework
from caretrace.evaluation.ragas_clinical_eval import (
    evaluate_ragas_clinical,
    evaluate_results_dataframe,
    summarize_ragas_scores,
    export_ragas_report,
)

print('✅ RAGAS framework imported')

## Step 1: Define Scenario Expectations

For each scenario, specify:
- `expected_fields`: Ground truth clinical data (temp, age, alertness, etc.)
- `expected_kg_topics`: What KG concepts should be retrieved

In [ ]:
# Define ground truth for each scenario
scenario_expectations = {
    'scenario_1_home': {
        'expected_fields': {
            'temp_f': 102.0,
            'age_years': 6.0,
            'alertness': 'normal',
            'fluid_intake': 'good',
            'urine_last_8h': 'yes',
            'breathing': 'normal',
        },
        'expected_kg_topics': ['fever', 'safety netting'],
    },
    'scenario_2_er': {
        'expected_fields': {
            'temp_f': 104.5,
            'age_years': 2.0,
            'alertness': 'altered',
            'fluid_intake': 'poor',
            'urine_last_8h': 'no',
            'breathing': 'distress',
        },
        'expected_kg_topics': ['fever', 'dehydration', 'altered mental status'],
    },
    'urgent_repeated_vomit': {
        'expected_fields': {
            'temp_f': 100.5,
            'age_years': 1.5,
            'alertness': 'normal',
            'fluid_intake': 'poor',
            'urine_last_8h': 'no',
            'breathing': 'normal',
        },
        'expected_kg_topics': ['vomiting', 'dehydration'],
    },
    'scenario_4_medication': {
        'expected_fields': {
            'temp_f': 101.0,
            'age_months': 5.0,
            'alertness': 'normal',
            'fluid_intake': 'good',
            'urine_last_8h': 'yes',
            'breathing': 'normal',
        },
        'expected_kg_topics': ['fever', 'ibuprofen age gate'],
    },
    'scenario_5_local_context': {
        'expected_fields': {
            'temp_f': 101.0,
            'age_years': 4.0,
            'alertness': 'normal',
            'fluid_intake': 'good',
            'urine_last_8h': 'yes',
            'breathing': 'normal',
        },
        'expected_kg_topics': ['fever', 'viral context'],
    },
}

print('✅ Scenario expectations defined')

## Step 2: Load Existing Results

Load results from `evaluation_full_metrics.ipynb` export

In [ ]:
# Load your existing evaluation results
results_path = Path('caretrace/evaluation/exports/results_full_metrics.csv')

if not results_path.exists():
    print(f"⚠️  Results file not found: {results_path}")
    print("    First run evaluation_full_metrics.ipynb and export results to CSV")
else:
    results_df = pd.read_csv(results_path)
    print(f'✅ Loaded {len(results_df)} results from {results_path.name}')
    print(f'   Columns: {list(results_df.columns)}')
    results_df.head()

## Step 3: Compute RAGAS Scores

Apply RAGAS metrics to all results

In [ ]:
# Add RAGAS scores to results
if 'results_df' in locals():
    ragas_results = evaluate_results_dataframe(
        results_df,
        scenario_expectations,
    )
    print(f'✅ Computed RAGAS scores for {len(ragas_results)} results')
    
    # Show RAGAS metrics
    ragas_cols = ['scenario_id', 'arm', 'expected', 'predicted', 
                  'extraction_accuracy', 'faithfulness', 'answer_relevance',
                  'context_relevance', 'context_recall', 'context_precision',
                  'ragas_score', 'clinical_score']
    ragas_results[ragas_cols].head(10)

## Step 4: Summarize by Arm

In [ ]:
if 'ragas_results' in locals():
    summary_arm = summarize_ragas_scores(ragas_results, by_arm=True)
    print('\n📊 RAGAS Scores by Arm\n')
    summary_arm.round(3)

## Step 5: Detailed Analysis

### Which arm performs best?

In [ ]:
if 'ragas_results' in locals():
    # Best arm by RAGAS score
    best_arm = ragas_results.groupby('arm')['ragas_score'].mean().sort_values(ascending=False)
    print('\n🏆 RAGAS Score by Arm (higher is better)\n')
    for arm, score in best_arm.items():
        print(f'{arm:45s}: {score:.3f}')
    
    # Best arm by clinical score
    best_clinical = ragas_results.groupby('arm')['clinical_score'].mean().sort_values(ascending=False)
    print('\n🏥 Clinical Score by Arm\n')
    for arm, score in best_clinical.items():
        print(f'{arm:45s}: {score:.3f}')

### Extraction Accuracy by Arm

In [ ]:
if 'ragas_results' in locals():
    extraction_by_arm = ragas_results.groupby('arm')['extraction_accuracy'].mean().sort_values(ascending=False)
    print('\n📊 Extraction Accuracy (critical foundation)\n')
    for arm, score in extraction_by_arm.items():
        pct = score * 100
        print(f'{arm:45s}: {score:.3f} ({pct:.1f}%)')

### Faithfulness Analysis

Is disposition grounded in rules + data?

In [ ]:
if 'ragas_results' in locals():
    faithfulness_by_arm = ragas_results.groupby('arm')['faithfulness'].mean().sort_values(ascending=False)
    print('\n🔗 Faithfulness Score (0-3)\n')
    for arm, score in faithfulness_by_arm.items():
        print(f'{arm:45s}: {score:.2f}/3.0')
    
    print('\nInterpretation:')
    print('  0.0 = No rule justification')
    print('  1.0 = Has rules but weak support')
    print('  2.0 = Good rule+data support')
    print('  3.0 = Excellent grounding')

### Answer Relevance

Is the response clinically appropriate and actionable?

In [ ]:
if 'ragas_results' in locals():
    relevance_by_arm = ragas_results.groupby('arm')['answer_relevance'].mean().sort_values(ascending=False)
    print('\n💬 Answer Relevance (0-3)\n')
    for arm, score in relevance_by_arm.items():
        print(f'{arm:45s}: {score:.2f}/3.0')

### KG Metrics (for arms with Neo4j enabled)

In [ ]:
if 'ragas_results' in locals():
    kg_arms = ragas_results[ragas_results['arm'].str.contains('kg_on', case=False)].copy()
    
    if len(kg_arms) > 0:
        print('\n📚 Knowledge Graph Metrics\n')
        print('Context Relevance (0-1: % of annotations relevant):')
        for arm in kg_arms['arm'].unique():
            score = kg_arms[kg_arms['arm']==arm]['context_relevance'].mean()
            print(f'  {arm:40s}: {score:.3f}')
        
        print('\nContext Recall (0-1: % of expected KG topics found):')
        for arm in kg_arms['arm'].unique():
            score = kg_arms[kg_arms['arm']==arm]['context_recall'].mean()
            print(f'  {arm:40s}: {score:.3f}')
        
        print('\nContext Precision (0-1: % of KG annotations high-confidence):')
        for arm in kg_arms['arm'].unique():
            score = kg_arms[kg_arms['arm']==arm]['context_precision'].mean()
            print(f'  {arm:40s}: {score:.3f}')
    else:
        print('⚠️  No KG-enabled arms found in results')

### Worst-Case Scenarios

Which cases had the lowest RAGAS scores?

In [ ]:
if 'ragas_results' in locals():
    worst = ragas_results.nsmallest(5, 'ragas_score')
    print('\n⚠️  Lowest RAGAS Scores (opportunities for improvement)\n')
    cols = ['scenario_id', 'arm', 'expected', 'predicted', 
            'extraction_accuracy', 'faithfulness', 'ragas_score']
    worst[cols]

## Step 6: Export Report

In [ ]:
if 'ragas_results' in locals():
    output_dir = Path('caretrace/evaluation/exports')
    export_ragas_report(ragas_results, output_dir)
    
    print('\n📄 Exported files:')
    print(f'  - {output_dir}/ragas_full_results.csv')
    print(f'  - {output_dir}/ragas_summary_by_arm.csv')
    print(f'  - {output_dir}/ragas_summary_by_scenario.csv')
    print(f'  - {output_dir}/ragas_report.md')

## Step 7: Custom Analysis

### Extraction Errors by Type

In [ ]:
if 'ragas_results' in locals():
    # Find cases where extraction failed
    extraction_failures = ragas_results[ragas_results['extraction_accuracy'] < 0.5]
    
    if len(extraction_failures) > 0:
        print(f'\n❌ {len(extraction_failures)} cases with extraction accuracy < 50%\n')
        cols = ['scenario_id', 'arm', 'extraction_accuracy', 'faithfulness']
        extraction_failures[cols].sort_values('extraction_accuracy')
    else:
        print('\n✅ No extraction failures detected')

### Disposition Accuracy vs RAGAS Score

Do correct dispositions have higher RAGAS scores?

In [ ]:
if 'ragas_results' in locals():
    correct = ragas_results[ragas_results['predicted'] == ragas_results['expected']]
    incorrect = ragas_results[ragas_results['predicted'] != ragas_results['expected']]
    
    print('\n📈 RAGAS Score vs Disposition Accuracy\n')
    print(f'Correct predictions ({len(correct)} cases):')
    print(f'  - Avg RAGAS score:    {correct["ragas_score"].mean():.3f}')
    print(f'  - Avg extraction acc: {correct["extraction_accuracy"].mean():.3f}')
    print(f'  - Avg faithfulness:   {correct["faithfulness"].mean():.2f}/3.0')
    
    print(f'\nIncorrect predictions ({len(incorrect)} cases):')
    print(f'  - Avg RAGAS score:    {incorrect["ragas_score"].mean():.3f}')
    print(f'  - Avg extraction acc: {incorrect["extraction_accuracy"].mean():.3f}')
    print(f'  - Avg faithfulness:   {incorrect["faithfulness"].mean():.2f}/3.0')

## Summary

### Overall System Performance

In [ ]:
if 'ragas_results' in locals():
    print('\n🎯 FINAL RAGAS METRICS\n')
    print(f'Overall RAGAS Score:        {ragas_results["ragas_score"].mean():.3f}/1.0')
    print(f'Overall Clinical Score:     {ragas_results["clinical_score"].mean():.3f}/1.0')
    print(f'Extraction Accuracy:        {ragas_results["extraction_accuracy"].mean():.3f}/1.0')
    print(f'Faithfulness:               {ragas_results["faithfulness"].mean():.2f}/3.0')
    print(f'Answer Relevance:           {ragas_results["answer_relevance"].mean():.2f}/3.0')
    print(f'Context Relevance:          {ragas_results["context_relevance"].mean():.3f}/1.0')
    print(f'Context Recall:             {ragas_results["context_recall"].mean():.3f}/1.0')
    print(f'Context Precision:          {ragas_results["context_precision"].mean():.3f}/1.0')
    
    print('\n✅ RAGAS evaluation complete! See exports/ for detailed reports.')